In [2]:
import pandas as pd

movies = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Documents\GenAI And AgenticAI\Moview Recomandation System\data\tmdb_5000_credits.csv")
credits = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Documents\GenAI And AgenticAI\Moview Recomandation System\data\tmdb_5000_movies.csv")

print(movies.head())
print(credits.head())

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                                cast  \
0  [{"cast_id": 242, "character": "Jake Sully", "...   
1  [{"cast_id": 4, "character": "Captain Jack Spa...   
2  [{"cast_id": 1, "character": "James Bond", "cr...   
3  [{"cast_id": 2, "character": "Bruce Wayne / Ba...   
4  [{"cast_id": 5, "character": "John Carter", "c...   

                                                crew  
0  [{"credit_id": "52fe48009251416c750aca23", "de...  
1  [{"credit_id": "52fe4232c3a36847f800b579", "de...  
2  [{"credit_id": "54805967c3a36829b5002c41", "de...  
3  [{"credit_id": "52fe4781c3a36847f81398c3", "de...  
4  [{"credit_id": "52fe479ac3a36847f813eaa3",

In [3]:
# merge data
movies = movies.merge(credits,on='title')

In [4]:
#select mportant columns
movies = movies[[
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "crew"
]]

In [6]:
#handle missing valus
movies.dropna(inplace=True)

In [7]:
#convert json
import ast
def convert(text):
    l=[]
    for i in ast.literal_eval(text):
        l.append(i['name'])
    return l


movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [8]:
#extract topp cast
def get_top_cast(text):
    l=[]
    counter = 0
    for i in ast.literal_eval(text):
        if counter !=3:
            l.append(i['name'])
            counter+=1
        else:
            break
    return l

movies['cast'] = movies['cast'].apply(get_top_cast)

In [9]:
# extract director
def fetch_director(text):
    l = []
    for i in ast.literal_eval(text):
        if i['job']=='director':
            l.append(i['name'])
    return l

movies['crew'] = movies['crew'].apply(fetch_director)

In [13]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [14]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [15]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [16]:
#combine everthins


In [11]:
#clean data
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [17]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies['tags'] = movies['tags'].apply(lambda x: x.lower())

In [18]:
print(type(movies['overview'][0]))
print(type(movies['genres'][0]))

<class 'list'>
<class 'list'>


In [19]:
print(movies[['title', 'tags']].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  in the 22nd century, a paraplegic marine is di...  
1  captain barbossa, long believed to be dead, ha...  
2  a cryptic message from bond’s past sends him o...  
3  following the death of district attorney harve...  
4  john carter is a war-weary, former military ca...  


# build Recommendation system

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vector = tfidf.fit_transform(movies['tags']).toarray()

print(vector.shape)

(4806, 5000)


In [23]:
# compute similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)

print(similarity.shape)

(4806, 4806)


# Recomadation function

In [24]:
def recommend(movie):
    
    movie = movie.lower()
    
    # find index
    idx = movies[movies['title'].str.lower() == movie].index
    
    if len(idx) == 0:
        return "Movie not found"
    
    idx = idx[0]
    
    distances = similarity[idx]
    
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]
    
    recommendations = []
    
    for i in movies_list:
        recommendations.append(movies.iloc[i[0]].title)
    
    return recommendations

In [25]:
print(recommend("Avatar"))

['Falcon Rising', 'Battle: Los Angeles', 'Apollo 18', 'Star Trek Into Darkness', 'Titan A.E.']


In [26]:
import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))